# Module D: Connectionist Networks

This notebook demonstrates the two connectionist models implemented in this module:
1. **Hopfield Network**: For pattern recognition and corruption recall.
2. **RNN Anomaly Detector**: For time-series anomaly detection on sensor data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from module_d_connectionist.hopfield import (
    train, recall, energy, corrupt,
    generate_pattern_1, generate_pattern_2, generate_pattern_3,
    generate_pattern_4, generate_pattern_5, generate_pattern_6
)
from module_d_connectionist.rnn_model import (
    AnomalyRNN, generate_synthetic_sequences
)

## 1. Hopfield Network

Generate patterns, corrupt one, and test recall.

In [ ]:
# 1. Pattern Generation
patterns = [
    generate_pattern_1(),
    generate_pattern_2(),
    generate_pattern_3(),
    generate_pattern_4(),
    generate_pattern_5(),
    generate_pattern_6()
]

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i, p in enumerate(patterns):
    axes[i].imshow(p.reshape(5, 5), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'Pattern {i+1}')
plt.show()

In [ ]:
# 2. Training and Corruption Recall Demo
W = train(patterns)

original = patterns[0]
corrupted = corrupt(original, flip_fraction=0.2)
recalled = recall(W, corrupted)

fig, axes = plt.subplots(1, 3, figsize=(6, 2))
axes[0].imshow(original.reshape(5, 5), cmap='gray'); axes[0].set_title("Original")
axes[1].imshow(corrupted.reshape(5, 5), cmap='gray'); axes[1].set_title("Corrupted (20%)")
axes[2].imshow(recalled.reshape(5, 5), cmap='gray'); axes[2].set_title("Recalled")
for ax in axes: ax.axis('off')
plt.show()

In [ ]:
# 3. Energy Plot during recall process
state = corrupted.copy()
energies = [energy(W, state)]
n_features = len(state)

for _ in range(20):
    indices = np.random.permutation(n_features)
    changed = False
    for i in indices:
        old_val = state[i]
        net_input = np.dot(W[i, :], state)
        new_val = 1 if net_input >= 0 else -1
        if new_val != old_val:
            state[i] = new_val
            changed = True
    energies.append(energy(W, state))
    if not changed:
        break

plt.plot(energies, marker='o')
plt.title('Hopfield Energy over Recall Iterations')
plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.show()

## 2. RNN Anomaly Detector

Train RNN on synthetic sequences and plot training curve.

In [ ]:
# RNN Training Curve & Accuracy
X, y = generate_synthetic_sequences(n=500)
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

model = AnomalyRNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []
epochs = 20
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))

plt.plot(losses, marker='o', color='orange')
plt.title('RNN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

model.eval()
with torch.no_grad():
    val_preds = (model(torch.tensor(X_val)) > 0.5).float()
    accuracy = (val_preds == torch.tensor(y_val)).float().mean().item()
print(f"Validation Accuracy: {accuracy * 100:.2f}%")